# Running VASP with the pyiron compatibility interface

`vaspparser.compatibility` provides the input options of the VASP job object of `pyiron_atomistics` as a single
function, `vasp_file_interface_function()`, without depending on `pyiron_base` or `pyiron_atomistics`. It is the VASP
equivalent of `lammpsparser.compatibility.file.lammps_file_interface_function()`, so VASP calculations can be used in
any Python-based workflow system.

* The arguments follow `pyiron_atomistics.vasp.vasp.vasp_function()`. Each setter of the job object becomes a
  `*_kwargs` argument with the arguments of that method, for example `job.set_kpoints(mesh=[4, 4, 4])` becomes
  `kpoints_kwargs={"mesh": [4, 4, 4]}`.
* Every `*_kwargs` argument accepts a plain dictionary or a dataclass from `vaspparser.compatibility.data`.
* The function returns `(shell_output, parsed_output, job_crashed)`. The parsed output is the dictionary of
  `vaspparser.vasp.output.parse_vasp_output()`.

This notebook runs as part of the continuous integration, where no VASP executable and no real pseudopotentials
are available:
* the POTCAR files come from the fake potential set in `static/vasp` (the test resources of `vaspparser`)
* the "VASP executable" is a small Python script that copies the output files of a finished VASP calculation
  (`static/vasp_test_files/full_job_sample`, a static calculation of bcc Fe with 2 atoms) into the working directory

So every step except the calculation itself is real: writing the input, executing a command, copying restart
files, parsing the output. On a machine with VASP, replace `vasp_command` with e.g. `"mpirun -np 4 vasp_std"` and
`resource_path` with the directory of your pyiron resources.

In [1]:
import os
import shutil
import sys
import warnings

import numpy as np
from ase.build import bulk, fcc111
from ase.constraints import FixAtoms

from vaspparser.compatibility.calculate import create_line_mode_structure
from vaspparser.compatibility.data import (
    CalcMDInput,
    CalcMinimizeInput,
    CalcStaticInput,
    CoulombInteractionsInput,
    KpointsInput,
    OccupancySmearingInput,
    SpinConstraintInput,
)
from vaspparser.compatibility.file import (
    collect_errors,
    get_vasp_input,
    restart_for_band_structure_calculations,
    restart_from_charge_density,
    restart_from_wave_and_charge,
    restart_from_wave_functions,
    vasp_file_interface_function,
)
from vaspparser.compatibility.input import get_potcar_files
from vaspparser.compatibility.output import check_convergence, structure_from_parsed_output

## Resources
The pseudopotentials are found from a pyiron-style resource directory. It contains `potentials_vasp.csv`, which
lists all potentials, `potentials_vasp_pbe_default.csv` / `potentials_vasp_lda_default.csv`, which give the default
potential per element, and the `potpaw_PBE` / `potpaw` directories with the POTCAR files. Pass it as `resource_path`,
or set the `VASPPARSER_RESOURCE_PATH` environment variable. The test resources contain (fake) POTCARs for `Fe` and
`Al_GW`.

In [2]:
RESOURCE_PATH = os.path.abspath("static/vasp")
SAMPLE_CALCULATION = os.path.abspath("static/vasp_test_files/full_job_sample")
WORKING_DIRECTORY = "compatibility_calculations"
shutil.rmtree(WORKING_DIRECTORY, ignore_errors=True)

for potcar in get_potcar_files(bulk("Fe", cubic=True), resource_path=RESOURCE_PATH) + get_potcar_files(
    bulk("Al", cubic=True), potentials={"Al": "Al_GW"}, resource_path=RESOURCE_PATH
):
    print(os.path.relpath(potcar))

static/vasp/potentials/potpaw_PBE/Fe/POTCAR
static/vasp/potentials/potpaw_PBE/Al_GW/POTCAR


## Generating the input files
`get_vasp_input()` takes the same arguments as `vasp_file_interface_function()` and returns the content of the input
files without running anything, the equivalent of inspecting `job.input` before `job.run()`. It is used in the
following to show which VASP tags each option sets.

In [3]:
def fe_bcc():
    """The structure of the sample calculation: bcc Fe, cubic cell with 2 atoms."""
    structure = bulk("Fe", "bcc", a=2.8, cubic=True)
    structure.set_initial_magnetic_moments(None)  # ase.build.bulk() adds magnetic moments for Fe
    return structure


def incar(**kwargs):
    """INCAR of the given settings as dictionary (VASP tag -> value string)."""
    text = get_vasp_input(resource_path=RESOURCE_PATH, **kwargs)["files_to_create"]["INCAR"]
    return dict(line.split("#")[0].strip().split("=", 1) for line in text.splitlines())


input_dict = get_vasp_input(
    structure=fe_bcc(),
    job_name="fe_static",
    plane_wave_cutoff=250,  # job.set_encut(250)
    kpoints_kwargs={"mesh": [4, 4, 4]},  # job.set_kpoints(mesh=[4, 4, 4])
    occupancy_smearing_kwargs={"smearing": "methfessel_paxton", "width": 0.1},  # job.set_occupancy_smearing(...)
    convergence_precision_kwargs={"electronic_energy": 1e-6},  # job.set_convergence_precision(...)
    calc_mode="static",  # job.calc_static()
    resource_path=RESOURCE_PATH,
)
for file_name in ["INCAR", "KPOINTS", "POSCAR"]:
    print(f"--- {file_name}\n{input_dict['files_to_create'][file_name]}")

--- INCAR
SYSTEM=fe_static #jobname
PREC=Accurate
ALGO=Fast
LREAL=.FALSE.
LWAVE=.FALSE.
LORBIT=0
ENCUT=250
EDIFFG=-0.01
EDIFF=1e-06
ISMEAR=1
SIGMA=0.1
IBRION=-1
NELM=100
NSW=0
LCHARG=.FALSE.
LVTOT=.FALSE.

--- KPOINTS
Kpoints file generated with pyiron_atomistics
0
Monkhorst_Pack
4 4 4
0 0 0

--- POSCAR
Poscar file generated with pyiron
1.0
2.800000000000000 0.000000000000000 0.000000000000000
0.000000000000000 2.800000000000000 0.000000000000000
0.000000000000000 0.000000000000000 2.800000000000000
Fe
2
Cartesian
0.000000000000000 0.000000000000000 0.000000000000000
1.400000000000000 1.400000000000000 1.400000000000000



Dictionaries and dataclasses are interchangeable:

In [4]:
input_dict_dataclasses = get_vasp_input(
    structure=fe_bcc(),
    job_name="fe_static",
    plane_wave_cutoff=250,
    kpoints_kwargs=KpointsInput(mesh=[4, 4, 4]),
    occupancy_smearing_kwargs=OccupancySmearingInput(smearing="methfessel_paxton", width=0.1),
    convergence_precision_kwargs={"electronic_energy": 1e-6},
    calc_dataclass=CalcStaticInput(),
    resource_path=RESOURCE_PATH,
)
assert input_dict_dataclasses == input_dict

### Calculation modes
`calc_mode` + `calc_kwargs` or `calc_dataclass` select the calculation, with the arguments of `calc_static()`,
`calc_minimize()` and `calc_md()` of the job object. The pyiron defaults are kept, e.g. `NELM = 400` for
minimisations and `ISYM = 0` for molecular dynamics.

In [5]:
keys = ["IBRION", "ISIF", "NSW", "NELM", "EDIFFG", "SMASS", "TEBEG", "TEEND", "POTIM", "NBLOCK", "ISYM", "LCHARG"]
for label, settings in {
    "relax positions": {"calc_dataclass": CalcMinimizeInput(ionic_force_tolerance=0.02)},
    "relax cell (p = 0)": {"calc_dataclass": CalcMinimizeInput(pressure=0.0, ionic_energy_tolerance=1e-4)},
    "relax volume only": {"calc_mode": "minimize", "calc_kwargs": {"volume_only": True}},
    "MD NVT": {"calc_dataclass": CalcMDInput(temperature=600, n_ionic_steps=100, time_step=2.0, n_print=10)},
    "MD NVE": {"calc_mode": "md", "calc_kwargs": {"temperature": None, "n_ionic_steps": 100}},
    "MD ramp": {"calc_mode": "md", "calc_kwargs": {"temperature": [300, 900], "retain_charge_density": True}},
}.items():
    tags = incar(structure=fe_bcc(), **settings)
    print(f"{label:20s}", {k: tags[k] for k in keys if k in tags})

assert incar(structure=fe_bcc(), calc_dataclass=CalcMinimizeInput(pressure=0.0))["ISIF"] == "3"
assert incar(structure=fe_bcc(), calc_mode="md", calc_kwargs={"temperature": 300})["ISYM"] == "0"

relax positions      {'IBRION': '2', 'ISIF': '2', 'NSW': '100', 'NELM': '400', 'EDIFFG': '-0.02', 'LCHARG': '.FALSE.'}
relax cell (p = 0)   {'IBRION': '2', 'ISIF': '3', 'NSW': '100', 'NELM': '400', 'EDIFFG': '0.0001', 'LCHARG': '.FALSE.'}
relax volume only    {'IBRION': '2', 'ISIF': '7', 'NSW': '100', 'NELM': '400', 'EDIFFG': '0.01', 'LCHARG': '.FALSE.'}
MD NVT               {'NSW': '100', 'SMASS': '3', 'TEBEG': '600', 'POTIM': '2.0', 'NBLOCK': '10', 'ISYM': '0', 'LCHARG': '.FALSE.'}
MD NVE               {'NSW': '100', 'SMASS': '-3', 'POTIM': '1.0', 'NBLOCK': '1', 'ISYM': '0', 'LCHARG': '.FALSE.'}
MD ramp              {'NSW': '1000', 'SMASS': '3', 'TEBEG': '300', 'TEEND': '900', 'POTIM': '1.0', 'NBLOCK': '1', 'ISYM': '0', 'LCHARG': '.TRUE.'}


### k-points
`kpoints_kwargs` covers all schemes of `job.set_kpoints()`: Monkhorst-Pack (`"MP"`), Gamma centred (`"GC"`), Gamma
point only (`"GP"`), manual k-points (`"Manual"`) and line mode (`"Line"`), plus `k_mesh_spacing` and
`symmetry_reduction=False` (`ISYM = -1`).

In [6]:
def kpoints(**kwargs):
    return get_vasp_input(structure=fe_bcc(), resource_path=RESOURCE_PATH, kpoints_kwargs=kwargs)["files_to_create"]["KPOINTS"]


print(kpoints(k_mesh_spacing=0.3, scheme="GC", center_shift=[0.5, 0.5, 0.5]))
print(kpoints(scheme="GP"))
print(kpoints(scheme="Manual", manual_kpoints=[[0, 0, 0], [0.5, 0, 0]], weights=[1, 3]))
assert incar(structure=fe_bcc(), kpoints_kwargs={"mesh": [2, 2, 2], "symmetry_reduction": False})["ISYM"] == "-1"

Kpoints file generated with pyiron_atomistics
0
Gamma centered
7 7 7
0.5 0.5 0.5

Kpoints file generated with pyiron_atomistics
0
Gamma Point
1 1 1
0 0 0

Kpoints file generated with pyiron_atomistics
2
Reciprocal
0 0 0 1
0.5 0 0 3



For band structures, `create_line_mode_structure()` returns the standardised primitive cell and the high symmetry
points and path from `seekpath` (the replacement for pyiron's `Atoms.create_line_mode_structure()`). Alternatively,
`band_structure_calc_kwargs` generates the path of pyiron's `job.set_for_band_structure_calc()`. Both need the
optional packages `seekpath` and `spglib` (`pip install vaspparser[compatibility]`).

In [7]:
fe_line, high_symmetry_points, high_symmetry_path = create_line_mode_structure(fe_bcc())
print("path:", high_symmetry_path["full"])
print(get_vasp_input(
    structure=fe_line,
    kpoints_kwargs={"scheme": "Line", "n_path": 20, "path_name": "full",
                    "high_symmetry_points": high_symmetry_points, "high_symmetry_path": high_symmetry_path},
    resource_path=RESOURCE_PATH,
)["files_to_create"]["KPOINTS"])

band_structure = get_vasp_input(structure=fe_bcc(), band_structure_calc_kwargs={"num_points": 10}, resource_path=RESOURCE_PATH)
tags = dict(line.split("#")[0].strip().split("=", 1) for line in band_structure["files_to_create"]["INCAR"].splitlines())
print({k: tags[k] for k in ["ICHARG", "ISYM"]}, "|", band_structure["files_to_create"]["KPOINTS"].splitlines()[1:4])

path: [('GAMMA', 'H'), ('H', 'N'), ('N', 'GAMMA'), ('GAMMA', 'P'), ('P', 'H'), ('P', 'N')]
Kpoints file generated with pyiron_atomistics
20
Line
rec
0.0 0.0 0.0 !GAMMA
0.5 -0.5 0.5 !H
0.5 -0.5 0.5 !H
0.0 0.0 0.5 !N
0.0 0.0 0.5 !N
0.0 0.0 0.0 !GAMMA
0.0 0.0 0.0 !GAMMA
0.25 0.25 0.25 !P
0.25 0.25 0.25 !P
0.5 -0.5 0.5 !H
0.25 0.25 0.25 !P
0.0 0.0 0.5 !N

{'ICHARG': '11', 'ISYM': '-1'} | ['19', 'Cartesian', '0.0 0.0 0.0 1.0']


### Electronic structure settings and output files

In [8]:
tags = incar(
    structure=fe_bcc(),
    plane_wave_cutoff=300,
    algorithm_kwargs={"algorithm": "Normal"},  # job.set_algorithm("Normal")
    mixing_parameters_kwargs={"method": "pulay", "n_pulay_steps": 20, "density_mixing_parameter": 0.3},
    fft_mesh_kwargs={"nx": 32, "ny": 32, "nz": 32},  # job.set_fft_mesh(...)
    n_empty_states=4,  # job.set_empty_states(4): NBANDS = N_electrons / 2 + 4
    occupancy_smearing_kwargs={"smearing": "fermi", "width": 0.05},
    write_wave_funct=True,  # job.write_wave_funct = True
    write_resolved_dos=11,  # job.write_resolved_dos = 11
    calc_kwargs={"retain_charge_density": True, "retain_electrostatic_potential": True},
)
print(tags)
assert tags["NBANDS"] == "12" and tags["LVHAR"] == ".TRUE." and tags["ISMEAR"] == "-1"

# non-default potential: job.potential.Al = "Al_GW"
print(get_vasp_input(structure=bulk("Al", a=4.04), potentials={"Al": "Al_GW"}, resource_path=RESOURCE_PATH)["files_to_create"]["POTCAR"])

{'SYSTEM': 'vasp_job', 'PREC': 'Accurate', 'ALGO': 'Normal', 'LREAL': '.FALSE.', 'LWAVE': '.TRUE.', 'LORBIT': '11', 'ENCUT': '300', 'ISMEAR': '-1', 'SIGMA': '0.05', 'NGXF': '32', 'NGYF': '32', 'NGZF': '32', 'IMIX': '4', 'MAXMIX': '20', 'AMIX': '0.3', 'NBANDS': '12', 'IBRION': '-1', 'NELM': '100', 'NSW': '0', 'LCHARG': '.TRUE.', 'LVTOT': '.TRUE.', 'LVHAR': '.TRUE.'}
# This is not a real POTCAR file because we don't have the rights to upload those
   12.00000000000000
# Just edit it with fake content to meet your testing needs



### Magnetism
The initial magnetic moments of the `ase` structure are written to `MAGMOM` and switch on `ISPIN = 2`, exactly like
in pyiron. Vector moments switch on `LNONCOLLINEAR` (use `vasp_ncl`), and `spin_constraint_kwargs` sets constrained
moments. DFT+U takes the U values per element.

In [9]:
afm = fe_bcc()
afm.set_initial_magnetic_moments([2.0, -2.0])
tags = incar(structure=afm)
print("AFM:", {k: tags[k] for k in ["ISPIN", "MAGMOM"]})

ncl = fe_bcc()
ncl.set_initial_magnetic_moments([[0, 0, 2], [0, 0, -2]])
tags = incar(structure=ncl, spin_constraint_kwargs=SpinConstraintInput(lamb=10.0, rwigs_dict={"Fe": 1.2}, direction=True))
print("constrained non-collinear:", {k: tags[k] for k in ["LNONCOLLINEAR", "MAGMOM", "M_CONSTR", "I_CONSTRAINED_M", "LAMBDA", "RWIGS", "ISYM"]})

tags = incar(structure=afm, coulomb_interactions_kwargs=CoulombInteractionsInput(ldauu={"Fe": 2.0}, ldaul={"Fe": 2}, ldauj={"Fe": 0.0}))
print("DFT+U:", {k: tags[k] for k in ["LDAU", "LDAUTYPE", "LDAUL", "LDAUU", "LDAUJ"]})
assert tags["LDAUU"] == "2.0"

AFM: {'ISPIN': '2', 'MAGMOM': '2.0   -2.0'}
constrained non-collinear: {'LNONCOLLINEAR': '.TRUE.', 'MAGMOM': '0.0 0.0 2.0   0.0 0.0 -2.0', 'M_CONSTR': '0.0 0.0 2.0   0.0 0.0 -2.0', 'I_CONSTRAINED_M': '1', 'LAMBDA': '10.0', 'RWIGS': '1.2', 'ISYM': '0'}
DFT+U: {'LDAU': '.TRUE.', 'LDAUTYPE': '2', 'LDAUL': '2', 'LDAUU': '2.0', 'LDAUJ': '0.0'}


### Constraints, surfaces and direct overrides
`FixAtoms` and `FixCartesian` constraints are written as selective dynamics. `dipole_correction_kwargs` and
`electric_field_kwargs` correspond to `job.set_dipole_correction()` and `job.set_electric_field()`. Any tag can be
set directly with `incar_file` (the equivalent of `job.input.incar[...] = ...`), and with `KSPACING` no KPOINTS file
is written.

In [10]:
slab = fcc111("Al", size=(1, 1, 3), a=4.04, vacuum=5.0)
slab.set_constraint(FixAtoms(indices=[0]))
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    files = get_vasp_input(
        structure=slab,
        potentials={"Al": "Al_GW"},
        dipole_correction_kwargs={"direction": 2, "dipole_center": [0.0, 0.0, 0.5]},
        incar_file={"NCORE": 2, "KSPACING": 0.3},
        resource_path=RESOURCE_PATH,
    )["files_to_create"]
print("warning:", [str(w.message) for w in caught if "KSPACING" in str(w.message)])
print(files["INCAR"])
print(files["POSCAR"])
assert "KPOINTS" not in files and "Selective dynamics" in files["POSCAR"]

SYSTEM=vasp_job #jobname
PREC=Accurate
ALGO=Fast
LREAL=.FALSE.
LWAVE=.FALSE.
LORBIT=11
ISYM=0
IDIPOL=3
LDIPOL=.TRUE.
EFIELD=0
DIPOL=0.0 0.0 0.5
IBRION=-1
NELM=100
NSW=0
LCHARG=.FALSE.
LVTOT=.FALSE.
NCORE=2
KSPACING=0.3

Poscar file generated with pyiron
1.0
2.856711395993652 0.000000000000000 0.000000000000000
1.428355697996826 2.473984640211010 0.000000000000000
0.000000000000000 0.000000000000000 14.664990175052177
Al
3
Selective dynamics
Direct
0.333333333333333 0.333333333333333 0.340948063402450 F F F
0.666666666666667 0.666666666666667 0.500000000000000 T T T
0.000000000000000 0.000000000000000 0.659051936597550 T T T



## Running a calculation
The stand-in for VASP: a script that copies the output of the sample calculation into the current directory.

In [11]:
fake_vasp = os.path.abspath(os.path.join(WORKING_DIRECTORY, "fake_vasp.py"))
os.makedirs(WORKING_DIRECTORY, exist_ok=True)
with open(fake_vasp, "w") as f:
    f.write(f"""import shutil
for name in ["vasprun.xml", "OUTCAR", "OSZICAR", "CONTCAR", "CHGCAR", "WAVECAR"]:
    shutil.copy({SAMPLE_CALCULATION!r} + "/" + name, name)
print("fake VASP: copied the output of the sample calculation")
""")
VASP_COMMAND = f'"{sys.executable}" "{fake_vasp}"'  # with VASP: "mpirun -np 4 vasp_std"

In [12]:
structure = fe_bcc()
shell_output, parsed_output, job_crashed = vasp_file_interface_function(
    working_directory=os.path.join(WORKING_DIRECTORY, "static"),
    structure=structure,
    job_name="fe_static",
    plane_wave_cutoff=250,
    kpoints_kwargs={"mesh": [4, 4, 4]},
    calc_mode="static",
    calc_kwargs={"retain_charge_density": True},
    write_wave_funct=True,
    vasp_command=VASP_COMMAND,
    resource_path=RESOURCE_PATH,
)
print(shell_output)
print("job crashed:", job_crashed)
print("parsed output:", list(parsed_output.keys()))
print("files:", sorted(os.listdir(os.path.join(WORKING_DIRECTORY, "static"))))

fake VASP: copied the output of the sample calculation

job crashed: False
parsed output: ['description', 'generic', 'structure', 'charge_density', 'electronic_structure', 'outcar']
files: ['CHGCAR', 'CONTCAR', 'INCAR', 'KPOINTS', 'OSZICAR', 'OUTCAR', 'POSCAR', 'POTCAR', 'WAVECAR', 'error.out', 'vasprun.xml']


The parsed output has the same structure as the output of the pyiron job object (`job.output` / `job["output/..."]`):

In [13]:
generic = parsed_output["generic"]
print("total energy (eV):", generic["energy_tot"])
print("forces (eV/A):\n", generic["forces"][-1])
print("pressure (eV/A^3):", generic["pressures"][-1])
print("Fermi level (eV):", parsed_output["electronic_structure"]["efermi"])
print("converged:", check_convergence(os.path.join(WORKING_DIRECTORY, "static"), parsed_output))

final_structure = structure_from_parsed_output(structure, parsed_output)  # job.get_structure()
print(final_structure)
assert len(generic["energy_tot"]) == 1
assert np.allclose(final_structure.cell, structure.cell)

total energy (eV): [-17.73798679]
forces (eV/A):
 [[-0.  0.  0.]
 [ 0. -0.  0.]]
pressure (eV/A^3): [-0.08177043 -0.08177043 -0.08177043]
Fermi level (eV): 5.97876516
converged: True
Atoms(symbols='Fe2', pbc=True, cell=[2.8, 2.8, 2.8])


## Restarts
The restart methods of the job object become helper functions. They take the working directory of the previous
calculation and return a `RestartInput`, which is passed as `restart_kwargs`. The restart files are copied into the
new working directory, and `ICHARG` / `ISTART` are set like in pyiron.

In [14]:
previous = os.path.join(WORKING_DIRECTORY, "static")
for helper in [restart_from_charge_density, restart_from_wave_functions, restart_from_wave_and_charge,
               restart_for_band_structure_calculations]:
    print(f"{helper.__name__:40s}", helper(previous))

restart_from_charge_density              RestartInput(charge_density_file='compatibility_calculations/static/CHGCAR', wave_function_file=None, icharg=1, self_consistent_calc=None, istart=None, restart_file_list=[])
restart_from_wave_functions              RestartInput(charge_density_file=None, wave_function_file='compatibility_calculations/static/WAVECAR', icharg=None, self_consistent_calc=None, istart=1, restart_file_list=[])
restart_from_wave_and_charge             RestartInput(charge_density_file='compatibility_calculations/static/CHGCAR', wave_function_file='compatibility_calculations/static/WAVECAR', icharg=1, self_consistent_calc=None, istart=1, restart_file_list=[])
restart_for_band_structure_calculations  RestartInput(charge_density_file='compatibility_calculations/static/CHGCAR', wave_function_file=None, icharg=11, self_consistent_calc=None, istart=None, restart_file_list=[])


In [15]:
restart_directory = os.path.join(WORKING_DIRECTORY, "restart")
shell_output, restart_output, job_crashed = vasp_file_interface_function(
    working_directory=restart_directory,
    structure=structure_from_parsed_output(structure, parsed_output),  # job.restart() continues from the final structure
    job_name="fe_restart",
    plane_wave_cutoff=250,
    kpoints_kwargs={"mesh": [4, 4, 4]},
    restart_kwargs=restart_from_wave_and_charge(previous),
    vasp_command=VASP_COMMAND,
    resource_path=RESOURCE_PATH,
)
with open(os.path.join(restart_directory, "INCAR")) as f:
    print(f.read())
assert "ISTART=1" in open(os.path.join(restart_directory, "INCAR")).read()
assert np.isclose(restart_output["generic"]["energy_tot"][-1], parsed_output["generic"]["energy_tot"][-1])

SYSTEM=fe_restart #jobname
PREC=Accurate
ALGO=Fast
LREAL=.FALSE.
LWAVE=.FALSE.
LORBIT=0
ENCUT=250
IBRION=-1
NELM=100
NSW=0
LCHARG=.FALSE.
LVTOT=.FALSE.
ISTART=1
ICHARG=1



## Failed calculations
With `accept_crash=True`, a failing executable returns `job_crashed=True` instead of raising an error.
`collect_errors()` checks the output for the EDDRMM and ZBRENT errors that the pyiron job object handles (set
`eddrmm_handling="warn"` to get them as warnings).

In [16]:
shell_output, crashed_output, job_crashed = vasp_file_interface_function(
    working_directory=os.path.join(WORKING_DIRECTORY, "crash"),
    structure=fe_bcc(),
    vasp_command=f'"{sys.executable}" -c "import sys; print(\'VASP failed\'); sys.exit(1)"',
    resource_path=RESOURCE_PATH,
    accept_crash=True,
)
print(shell_output, "job crashed:", job_crashed, "| parsed output:", crashed_output)
assert job_crashed and crashed_output is None

with open(os.path.join(WORKING_DIRECTORY, "crash", "error.out"), "a") as f:
    f.write(" WARNING in EDDRMM: call to ZHEGV failed, returncode = 6\n")
print(collect_errors(os.path.join(WORKING_DIRECTORY, "crash")))

VASP failed
 job crashed: True | parsed output: None


['EDDRMM warnings occured 1 times, first in ionic step 0.']


## Using it in workflows
The function needs only serialisable inputs (an `ase` structure, numbers, dictionaries or dataclasses) and returns
plain Python objects, so it can be submitted with any workflow or task-scheduling framework. For example, it can be
wrapped with the `job` decorator of `pyiron_base` to store the result in a pyiron project:

```python
from pyiron_base import Project, job

pr = Project("demo")
vasp_job = job(vasp_file_interface_function)(
    working_directory="vasp_calc", structure=fe_bcc(), plane_wave_cutoff=250, vasp_command="mpirun -np 4 vasp_std",
    pyiron_project=pr,
)
shell_output, parsed_output, job_crashed = vasp_job.pull()
```

In [17]:
shutil.rmtree(WORKING_DIRECTORY)